# CA15 Quick Demo

This notebook demonstrates how to import and run core components from CA15. It is intentionally non-executed by default and provides a step-by-step template for experiments and debugging.

## 1 — Import and Environment Setup

Import standard libraries and project modules. All imports are written to be import-safe (no side-effects at import time).

In [ ]:
# Standard libs
import sys
import platform
from pathlib import Path

# Add project `src` to PYTHONPATH (when opening the notebook from repo root)
ROOT = Path("../").resolve()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

# Project modules (import-safe)
import config as cfg_mod
import data as data_mod
import model as model_mod
import losses as losses_mod
import train as train_mod
import utils as utils_mod

print("Python:", platform.python_version())
print("Numpy:", np.__version__)
try:
    import torch
    print("PyTorch:", torch.__version__)
except Exception:
    print("PyTorch not available in this environment")

## 2 — Load Configuration

Load config from `configs/default.yaml` and override a couple of keys for a short demo run.

In [ ]:
from config import Config

cfg = Config.from_yaml("../configs/default.yaml")
print(cfg)

# Example override for a quick demo
cfg.epochs = 1
cfg.batch_size = 16
print("Overridden for demo:", cfg)

## 3 — Create or Load Dataset

Create a small synthetic dataset and visualize a couple of samples.

In [ ]:
from data import SyntheticDataset

ds = SyntheticDataset(cfg.input_dim, cfg.output_dim, size=128, seed=cfg.seed)
print("dataset size:", len(ds))

# show a few samples
s, a, v = next(ds.batches(5))
print("states.shape", s.shape)
print("actions", a)
print("values", v)

plt.figure()
plt.plot(s.numpy()[0])
plt.title("Example state features")
plt.show()

## 4 — Instantiate Model

Create policy and value networks and check parameter counts and a forward pass.

In [ ]:
from model import MLPPolicy, ValueNetwork

policy = MLPPolicy(cfg.input_dim, cfg.hidden_dim, cfg.output_dim)
value = ValueNetwork(cfg.input_dim, cfg.hidden_dim)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("policy params:", count_params(policy))
print("value params:", count_params(value))

# forward pass check
x = np.random.randn(2, cfg.input_dim).astype(np.float32)
import torch
x_t = torch.from_numpy(x)
print("policy output shape:", policy(x_t).shape)
print("value output shape:", value(x_t).shape)

## 5 — Losses and Metrics

Use the provided losses and compute a simple metric.

In [ ]:
from losses import mse_loss, policy_gradient_loss

# MSE example
pred = value(x_t)
target = torch.randn_like(pred)
print("MSE:", mse_loss(pred, target).item())

# Policy loss example (toy)
logp = torch.log_softmax(policy(x_t), dim=-1).gather(-1, torch.randint(0, cfg.output_dim, (2,1))).squeeze(-1)
adv = torch.randn(2)
print("policy loss:", policy_gradient_loss(logp, adv).item())

## 6 — Single Training Step (manual)

A single manual forward/backward update using `torch` and `optim`.

In [ ]:
import torch.optim as optim

opt_v = optim.Adam(value.parameters(), lr=cfg.lr)
opt_p = optim.Adam(policy.parameters(), lr=cfg.lr)

s_batch, a_batch, v_batch = next(ds.batches(cfg.batch_size))
opt_v.zero_grad()
val_preds = value(s_batch)
val_loss = mse_loss(val_preds, v_batch)
val_loss.backward()
opt_v.step()

opt_p.zero_grad()
logits = policy(s_batch)
probs = torch.nn.functional.softmax(logits, dim=-1)
dist = torch.distributions.Categorical(probs=probs)
logp = dist.log_prob(a_batch)
adv = (v_batch - val_preds).detach()
pol_loss = policy_gradient_loss(logp, adv)
pol_loss.backward()
opt_p.step()

print("val_loss", val_loss.item(), "pol_loss", pol_loss.item())

## 7 — Training Loop (using `src.train`)

Use the `train` function for short experiments.

In [ ]:
from train import train

# run a short training run (not executed here)
# summary = train(cfg)
# print(summary)

print("To run: from terminal use: python -m src.train --cfg configs/default.yaml")

In [ ]:
from utils import save_checkpoint, load_checkpoint

# save example (not executed here)
# save_checkpoint('checkpt.pt', policy, opt_p, extra={'cfg': cfg.__dict__})
# payload_extra = load_checkpoint('checkpt.pt', policy, opt_p)

print("See `save_checkpoint` and `load_checkpoint` in src.utils for API details")